# Rizzler V24 — Context Convergence & Multi-Agent Live Qualification

**Production acceptance exercise**, not a certificate of production readiness. Run on Kaggle with Internet and a paid Gemini API key added as a Kaggle Secret named `GEMINI_API_KEY`. A GPU T4 may be enabled, but Gemini inference runs through the hosted API. Do not expose tokens in cell output.

This notebook pins the Rizzler V24 PR56 source at commit `622a3282878ff29e19f43a360cd807a602951905`, validates local contracts, performs real model preflight, creates an actual multi-format SQL/Excel/Word/PPT/PDF job, and separately attempts a real source-research workflow. It saves diagnostics and rendered artifacts only on completion. If anything fails, the notebook **reports the failure** and preserves evidence rather than inventing a successful result.

Run cells in order. Internet access, Gemini model compatibility, billing and rate limits determine whether live qualification can finish.

In [ ]:
# 1. Configure run isolation and read Kaggle Secret securely
import os, sys, json, subprocess, pathlib, time, secrets, hashlib, datetime
BASE=pathlib.Path("/kaggle/working/rizzler_v24_qualification")
BASE.mkdir(parents=True,exist_ok=True)
SRC=BASE/"Voice-AI"
PIN="622a3282878ff29e19f43a360cd807a602951905"
BRANCH="feature/v24-context-convergence-and-safe-repair"
from kaggle_secrets import UserSecretsClient
api_key=UserSecretsClient().get_secret("GEMINI_API_KEY")
assert isinstance(api_key,str) and len(api_key)>15, "Kaggle Secret GEMINI_API_KEY is missing"
os.environ["GEMINI_API_KEY"]=api_key
os.environ["RIZZLER_ACCESS_KEY"]=secrets.token_urlsafe(32)
os.environ["RIZZLER_WORK_ROOT"]=str(BASE/"work")
os.environ["RIZZLER_V23_MODE"]="full"
os.environ["RIZZLER_GEMINI_ENABLE_THINKING"]="true"
os.environ["RIZZLER_GEMINI_MODEL"]=os.environ.get("RIZZLER_GEMINI_MODEL","gemini-3.5-flash-lite")
os.environ["RIZZLER_PROVIDER_SLOT_ROOT"]=str(BASE/"rate_governor")
os.environ["RIZZLER_GEMINI_RPM"]=os.environ.get("RIZZLER_GEMINI_RPM","12")
print("Run directory:",BASE)
print("Model configured:",os.environ["RIZZLER_GEMINI_MODEL"])
print("API key:", "present (redacted)")

In [ ]:
# 2. Fetch exactly the tested revision and install Work/voice-independent artifact extras
if not SRC.exists():
    subprocess.run(["git","clone","--filter=blob:none","--no-checkout",
                    "https://github.com/SAUNAK359/Voice-AI.git",str(SRC)],check=True)
subprocess.run(["git","-C",str(SRC),"fetch","--depth","1","origin",PIN],check=True)
subprocess.run(["git","-C",str(SRC),"checkout","--force",PIN],check=True)
rev=subprocess.check_output(["git","-C",str(SRC),"rev-parse","HEAD"],text=True).strip()
assert rev==PIN, "Unexpected checkout revision"
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(SRC)+"[work,test]"],check=True)
print("Pinned source verified:",rev)

In [ ]:
# 3. Minimal offline contracts, isolated from paid Gemini
env=os.environ.copy()
env["RIZZLER_V23_MODE"]="full"
tests=subprocess.run([
    sys.executable,"-m","pytest","-q",
    str(SRC/"tests/test_v24_context_convergence.py"),
    str(SRC/"tests/test_v24_intent_and_recovery.py"),
    str(SRC/"tests/test_v23_flags.py")],
    cwd=str(SRC),env=env,text=True,capture_output=True)
(BASE/"v24_contract_tests.txt").write_text(tests.stdout+"\n"+tests.stderr)
print(tests.stdout[-3000:])
assert tests.returncode==0, "Local V24 contract suite failed; examine v24_contract_tests.txt"

In [ ]:
# 4. Real Gemini compatibility/billing preflight
from google import genai
from google.genai import types
from rizzler.config import Settings
from rizzler.v23_flags import enabled
settings=Settings.from_env()
assert enabled("RIZZLER_CONTEXT_CONVERGENCE") and enabled("RIZZLER_GEMINI_NATIVE_TOOLS")
assert settings.gemini_enable_thinking, "V24 default thinking unexpectedly disabled"
client=genai.Client(api_key=api_key)
model=settings.gemini_model
try:
    model_info=client.models.get(model=model)
    print("Gemini model lookup succeeded:",model)
    response=client.models.generate_content(
        model=model,contents="Reply with the exact text: V24_API_OK",
        config=types.GenerateContentConfig(temperature=0,max_output_tokens=40))
    assert "V24_API_OK" in (response.text or ""), "Gemini model text validation failed"
    print("Gemini inference:", response.text.strip())
finally:
    client.close()

## Live multi-format job: synthetic audited data

The next job uses an explicitly **synthetic** three-year trade dataset with exact arithmetic. Do not treat these values as actual Indian trade statistics. This verifies source SQL, multi-agent cooperation, PowerPoint/Word/Excel/PDF generation and final artifact checks without mixing fabricated data into external research.

In [ ]:
# 5. Create audited synthetic CSV and register it in an isolated WorkStore
import csv, io
from rizzler.work_store import WorkStore
from rizzler.work_worker import WorkWorker
store=WorkStore(BASE/"work")
owner="kaggle-v24-qualification"
table=[
    {"year":"2023","exports_usd_mn":80,"imports_usd_mn":110},
    {"year":"2024","exports_usd_mn":95,"imports_usd_mn":115},
    {"year":"2025","exports_usd_mn":125,"imports_usd_mn":150},
]
csv_io=io.StringIO()
writer=csv.DictWriter(csv_io,fieldnames=list(table[0]))
writer.writeheader()
writer.writerows(table)
uploaded=store.register_upload(owner,"synthetic_trade.csv",csv_io.getvalue().encode())
print("Synthetic data:",table)
print("Dataset upload ID:",uploaded["id"])

In [ ]:
# 6. Submit the actual durable Work Agent multi-artifact task
formats=["data_analysis","pptx","docx","xlsx","pdf"]
objective=(
 "Analyze only the attached clearly labeled SYNTHETIC dataset, with exact "
 "imports, exports, trade balance (exports minus imports), year over year "
 "changes and limitations. Produce a professional editable presentation, "
 "Word report, Excel workbook with checked formulas where appropriate and "
 "a PDF report. Use USD millions exactly as present in the dataset; no "
 "unstated external sources or invented current economic figures. "
 "Verify cross-format consistency, evidence dependencies and output quality."
)
job=store.submit(owner,objective,formats,
                 "v24-synthetic-multiformat-001",[uploaded["id"]])
print("Created multi-format job",job["id"],job["status"])

In [ ]:
# 7. Run one actual worker claim (may use many paid Gemini requests)
import asyncio
start=time.monotonic()
ran=asyncio.run(WorkWorker(store).once())
elapsed=round(time.monotonic()-start,2)
multi=store.get(owner,job["id"])
print("Worker claimed:",ran,"elapsed_s:",elapsed)
print("Job result:",multi["status"],multi["stage"],multi["error_code"])
print("Agents:",[(a["agent"],a["status"],a["attempt"],a["error_code"])
                  for a in store.agent_steps(owner,job["id"])])
if multi["status"]=="awaiting_input":
    print("Unresolved owner questions:",store.questions(owner,job["id"]))
print("Strict quality report:",str(store.quality_report(owner,job["id"]))[:2500])
(BASE/"synthetic_status.json").write_text(json.dumps({
    "job":multi,"seconds":elapsed,
    "agents":store.agent_steps(owner,job["id"]),
    "questions":store.questions(owner,job["id"]),
    "events":store.events(owner,job["id"]),
    "diagnostics":store.diagnostics(owner,job["id"])},indent=2,default=str))
assert multi["status"]=="completed", "Multi-format workflow not qualified; status/diagnostics saved"

In [ ]:
# 8. Independently open real artifact bytes and inspect basic content
from pptx import Presentation
from docx import Document
from openpyxl import load_workbook
from pypdf import PdfReader
assert multi["status"]=="completed"
all_artifacts=store.artifacts(owner,job["id"])
assert {x["kind"] for x in all_artifacts} == {"pptx","docx","xlsx","pdf"},all_artifacts
for receipt in all_artifacts:
    path,record=store.artifact(owner,receipt["id"])
    data=path.read_bytes()
    assert hashlib.sha256(data).hexdigest()==receipt["sha256"]
    if receipt["kind"]=="pptx":
        p=Presentation(str(path))
        assert len(p.slides)>=3
        print("PPTX:",len(p.slides),"editable slides")
    if receipt["kind"]=="docx":
        d=Document(str(path))
        assert len(d.paragraphs)>=3
        print("DOCX:",len(d.paragraphs),"paragraphs")
    if receipt["kind"]=="xlsx":
        w=load_workbook(str(path),data_only=False)
        assert len(w.worksheets)>=1
        print("XLSX sheets:",w.sheetnames)
    if receipt["kind"]=="pdf":
        reader=PdfReader(str(path))
        assert len(reader.pages)>=1
        print("PDF pages:",len(reader.pages))
print("All requested real artifacts verified structurally.")

## Separate real grounded research workflow

This is not a mock: the model must perform real Google Search and the existing Rizzler verified-quote/source-retrieval gates. Unavailable or inaccessible authoritative sources are reported, not treated as successful evidence. Output review and cross-format consistency must be inspected before a production claim.

In [ ]:
# 9. Submit an independent real research job
research_objective=(
 "Investigate India's merchandise imports and exports for fiscal year "
 "2024-25 using official public sources. Compare total exports versus total "
 "imports and their balance, explicitly distinguish goods and services, "
 "period definitions and currency units. Cite the source URLs you actually "
 "retrieved; identify uncertainty and gaps. Prepare a sourced PPTX, Word "
 "report and PDF for an analytical stakeholder. No fabricated figures."
)
research_job=store.submit(owner,research_objective,
                           ["research","pptx","docx","pdf"],
                           "v24-research-live-001",[])
print("Research job:",research_job["id"],research_job["status"])

In [ ]:
# 10. Run research + deliverables live and capture transparent outcome
t0=time.monotonic()
claimed=asyncio.run(WorkWorker(store).once())
research_status=store.get(owner,research_job["id"])
print("Worker claimed:",claimed,"elapsed_s:",round(time.monotonic()-t0,2))
print("Research result:",research_status["status"],research_status["error_code"])
print("Agent stages:",[(a["agent"],a["status"],a["error_code"])
                      for a in store.agent_steps(owner,research_job["id"])])
print("User questions:",store.questions(owner,research_job["id"]))
print("Private diagnostics:",store.diagnostics(owner,research_job["id"]))
(BASE/"research_status.json").write_text(json.dumps({
    "job":research_status,"agents":store.agent_steps(owner,research_job["id"]),
    "questions":store.questions(owner,research_job["id"]),
    "events":store.events(owner,research_job["id"]),
    "diagnostics":store.diagnostics(owner,research_job["id"])},indent=2,default=str))
if research_status["status"]!="completed":
    print("NOT QUALIFIED: research/format workflow did not complete successfully.")
else:
    print("Research workflow generated:",[x["kind"] for x in store.artifacts(owner,research_job["id"])])

In [ ]:
# 11. Export a redacted qualification evidence ZIP, including generated artifacts
import zipfile
pack=BASE/"Rizzler_V24_Live_Qualification_Evidence.zip"
with zipfile.ZipFile(pack,"w",zipfile.ZIP_DEFLATED) as z:
    for name in ("v24_contract_tests.txt","synthetic_status.json","research_status.json"):
        p=BASE/name
        if p.exists():z.write(p,arcname=name)
    manifest={"source_sha":PIN,"model_id":model,
              "mode":"full","thinking":settings.gemini_enable_thinking,
              "synthetic_job_status":multi["status"],
              "research_job_status":research_status["status"],
              "created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
              "limitations":"No private model prompts/API keys; domain facts and visual quality require independent review"}
    z.writestr("MANIFEST.json",json.dumps(manifest,indent=2))
    for job_id in (job["id"],research_job["id"]):
        for row in store.artifacts(owner,job_id):
            p,_=store.artifact(owner,row["id"])
            z.write(p,arcname=f"artifacts/{job_id}/{p.name}")
print("Evidence bundle:",pack,"bytes:",pack.stat().st_size)
print("Release status:", "NEEDS_REVIEW" if multi["status"]=="completed" and research_status["status"]=="completed" else "NOT_QUALIFIED")

### Release conclusion

Even if all cells pass, perform human/independent visual review of the generated PPTX/DOCX/PDF, verify Excel formulas against input data, trace every factual research claim to the retrieved primary source, repeat under provider timeouts and test job recovery across process restarts. **Green tests are not an automatic certification.** Download the Evidence ZIP from Kaggle Output and compare its manifest and diagnostics with the PR56 source.